# Bank Marketing

https://archive.ics.uci.edu/dataset/222/bank+marketing

In [9]:
import pandas as pd
import numpy as np
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import recall_score

from sklearn.model_selection import learning_curve
from sklearn.inspection import permutation_importance

In [4]:
%pip install ucimlrepo

  Using cached ucimlrepo-0.0.7-py3-none-any.whl.metadata (5.5 kB)
Using cached ucimlrepo-0.0.7-py3-none-any.whl (8.0 kB)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# metadata
print(bank_marketing.metadata)

# variable information
print(bank_marketing.variables)


{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [13]:
X.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [7]:
y.head()

,y
0,no
1,no
2,no
3,no
4,no


In [10]:
X.shape, y.shape

((45211, 16), (45211, 1))

In [16]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
dtypes: int64(7), object(9)
memory usage: 5.5+ MB


In [11]:
y.value_counts(normalize=True)

y  
no     0.883015
yes    0.116985
Name: proportion, dtype: float64

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((31647, 16), (13564, 16), (31647, 1), (13564, 1))

In [21]:
num_col = [
    "age",
    "balance",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

cat_col = [
    "job",
    "marital",
    "education",
    "contact",
    "default",
    "housing",
    "loan",
    "day_of_week",
    "month",
    "poutcome"
]

num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor= ColumnTransformer([
    ('num_transformer', num_transformer, num_col),
    ('cat_transformer', cat_transformer, cat_col)
])

In [22]:
preprocessor

ColumnTransformer(transformers=[('num_transformer', StandardScaler(),
                                 ['age', 'balance', 'duration', 'campaign',
                                  'pdays', 'previous']),
                                ('cat_transformer',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'contact',
                                  'default', 'housing', 'loan', 'day_of_week',
                                  'month', 'poutcome'])])

In [23]:
pipeline_lr = make_pipeline(preprocessor, LogisticRegression())
pipeline_lr

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_transformer',
                                                  StandardScaler(),
                                                  ['age', 'balance', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_transformer',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'default', 'housing', 'loan',
                                                   'day_of_week', 'month',
                                                   'poutcome'])])),
                ('logisticregression', LogisticRegression())])

In [24]:
cross_val_score(pipeline_lr, X_train, y_train, cv=5).mean()

/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-pack

0.9004644434942641

In [25]:
pipeline_lr.fit(X_train, y_train)

/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_transformer',
                                                  StandardScaler(),
                                                  ['age', 'balance', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_transformer',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'default', 'housing', 'loan',
                                                   'day_of_week', 'month',
                                                   'poutcome'])])),
                ('logisticregression', LogisticRegression())])

In [26]:
pipeline_lr.score(X_test, y_test)

0.9057800058979653

In [27]:
pipeline_knn = make_pipeline(preprocessor, KNeighborsClassifier())
pipeline_tree = make_pipeline(preprocessor, DecisionTreeClassifier())
pipeline_random = make_pipeline(preprocessor, RandomForestClassifier())
pipeline_svc = make_pipeline(preprocessor, SVC())

In [28]:
print(cross_val_score(pipeline_knn, X_train, y_train, cv=5).mean())
print(cross_val_score(pipeline_tree, X_train, y_train, cv=5).mean())
print(cross_val_score(pipeline_random, X_train, y_train, cv=5).mean())
print(cross_val_score(pipeline_svc, X_train, y_train, cv=5).mean())

/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-package

0.8938603289803924
0.8754697813944539


/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/sit

0.9028975175581596


/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-pack

0.9019812108908638


In [29]:
pipeline_knn.fit(X_train, y_train)
pipeline_tree.fit(X_train, y_train)
pipeline_random.fit(X_train, y_train)
pipeline_svc.fit(X_train, y_train)

/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_transformer',
                                                  StandardScaler(),
                                                  ['age', 'balance', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_transformer',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'default', 'housing', 'loan',
                                                   'day_of_week', 'month',
                                                   'poutcome'])])),
                ('svc', SVC())])

In [30]:
print(pipeline_knn.score(X_test, y_test))
print(pipeline_tree.score(X_test, y_test))
print(pipeline_random.score(X_test, y_test))
print(pipeline_svc.score(X_test, y_test))

0.9006930109112357
0.8771011501032144
0.9074019463285167
0.9074756708935418


In [31]:
pipeline_svc.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('num_transformer', StandardScaler(),
                                    ['age', 'balance', 'duration', 'campaign',
                                     'pdays', 'previous']),
                                   ('cat_transformer',
                                    OneHotEncoder(handle_unknown='ignore'),
                                    ['job', 'marital', 'education', 'contact',
                                     'default', 'housing', 'loan', 'day_of_week',
                                     'month', 'poutcome'])])),
  ('svc', SVC())],
 'transform_input': None,
 'verbose': False,
 'columntransformer': ColumnTransformer(transformers=[('num_transformer', StandardScaler(),
                                  ['age', 'balance', 'duration', 'campaign',
                                   'pdays', 'previous']),
                                 ('cat_transformer',
                                  OneHo

In [33]:
params = {
    'svc__C': uniform(0.1, 10),
    'svc__gamma': uniform(0.001, 0.1)
}

search_svm = RandomizedSearchCV(
    pipeline_svc,
    params,
    n_iter=3,
    n_jobs=-1,
    cv=3,
    verbose=1,
    random_state=42
)

search_svm.fit(X_train, y_train)
search_svm.best_params_

Fitting 3 folds for each of 3 candidates, totalling 9 fits


/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-pack

{'svc__C': 3.845401188473625, 'svc__gamma': 0.09607143064099162}

In [34]:
search_svm.best_estimator_

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_transformer',
                                                  StandardScaler(),
                                                  ['age', 'balance', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_transformer',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'default', 'housing', 'loan',
                                                   'day_of_week', 'month',
                                                   'poutcome'])])),
                ('svc', SVC(C=3.845401188473625, gamma=0.09607143064099162))])

In [35]:
clf = search_svm.best_estimator_
print(cross_val_score(clf, X_train, y_train, cv=5).mean())

/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/yaeyukimoto/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-pack

0.9034031066903596
